In [ ]:
# Copyright (C) 2024 Richard Stiskalek
# This program is free software; you can redistribute it and/or modify it
# under the terms of the GNU General Public License as published by the
# Free Software Foundation; either version 3 of the License, or (at your
# option) any later version.
#
# This program is distributed in the hope that it will be useful, but
# WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU General
# Public License for more details.
#
# You should have received a copy of the GNU General Public License along
# with this program; if not, write to the Free Software Foundation, Inc.,
# 51 Franklin Street, Fifth Floor, Boston, MA  02110-1301, USA.
from os.path import exists
import numpy as np
import matplotlib.pyplot as plt
from corner import corner
from getdist import plots
import scienceplots


from reconstruction_comparison import *

%load_ext autoreload
%autoreload 2
%matplotlib inline

paths = csiborgtools.read.Paths(**csiborgtools.paths_glamdring)
fdir = "/mnt/extraspace/rstiskalek/csiborg_postprocessing/peculiar_velocity"

### Quick checks

In [ ]:
fname = "/mnt/extraspace/rstiskalek/catalogs/IndranilVoid/ChiSq2D_EXPprofileresolution_3501x601_samples.hdf5"

# samples = get_some_samples(fname, [ "sigma_v", "aFP", "bFP", "cFP", "e_mu", "beta", "rLG", "void_size"])
samples = get_some_samples(fname, ["rLG", "Vvoid"])

print("Read in samples are:\n", list(samples.keys()))
data, labels, keys = samples_for_corner(samples)


fig = corner(data, labels=labels, show_titles=True,
             title_kwargs={"fontsize": 12}, smooth=1)

fig.savefig("../../plots/exp_test.png", dpi=450)
fig.show()

### Sergij's $\chi^2$ grid

In [ ]:
kind = "MB"
fpath = f"/mnt/extraspace/rstiskalek/catalogs/IndranilVoid/ChiSq2D_{kind}profileresolution_3501x601.dat"  # noqa
print(f"Reading the chi2 grid from `{fpath}`.")

data = np.genfromtxt(fpath)
nx, ny = data.shape
xdata = np.arange(0, nx)
ydata = np.arange(0, ny)

In [ ]:
i, j = np.unravel_index(np.argmin(data), data.shape)
print(xdata[i], ydata[j])

In [ ]:
plt.figure()
plt.imshow(np.exp(-0.5 * data), origin="lower", aspect="auto")
plt.show()

## Paper plots

### Fiducial analysis

- Keeping the void size fixed but varying the observer position, along with the external velocity



#### Corner plot of the posterior

In [ ]:
catalogue = ['LOSS', 'Foundation', 'CF4_TFR_i', 'CF4_TFR_notSDSS_w1']
# catalogue = ['LOSS', 'Foundation']
ranges = {"rLG": (0, None)}

X = []
for simname in ["IndranilVoid_exp"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=False, sample_beta=None,
        zcmb_max=0.05)

    X_i = samples_to_getdist(get_samples(fname, True), simname_to_pretty(simname), ranges=ranges)
    X.append(X_i)

fname = "/mnt/extraspace/rstiskalek/catalogs/IndranilVoid/ChiSq2D_EXPprofileresolution_3501x601_samples.hdf5"
samples = get_samples(fname, False)
samples["Vmag"] = samples.pop("Vvoid")
X_i = samples_to_getdist(samples, "Exponential (BF-constrained)")
X.append(X_i)

# fname = "/mnt/extraspace/rstiskalek/catalogs/IndranilVoid/ChiSq2D_GAUSSprofileresolution_3501x601_samples.hdf5"
# samples = get_samples(fname, False)
# samples["Vmag"] = samples.pop("Vvoid")
# X_i = samples_to_getdist(samples, "Gauss (BF-constrained)")
# X.append(X_i)



params = ["Vmag", "l", "b", "rLG"]
with plt.style.context("science"):
    g = plots.get_subplot_plotter()
    g.settings.figure_legend_frame = False
    g.settings.alpha_filled_add = 0.75
    g.settings.legend_fontsize = 14  # Change this to your desired font size

    g.triangle_plot(X, params=params, filled=True, legend_loc='upper right')
    # plt.gcf().suptitle(simname_to_pretty(simname), y=1.025)
    plt.gcf().tight_layout()
    plt.gcf().show()
    plt.gcf().savefig(f"../../plots/void_fiducial.pdf", dpi=500, bbox_inches='tight')

#### Evidence comparison

In [ ]:
catalogue = ['LOSS', 'Foundation', 'CF4_TFR_i', 'CF4_TFR_notSDSS_w1']

X = {}
for simname in ["IndranilVoid_exp", "IndranilVoid_gauss", "IndranilVoid_mb", "no_field"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=False, sample_beta=None, no_Vext=None,
        zcmb_max=0.05)

    X[simname] = -get_gof("neg_lnZ_harmonic", fname)


for simname in ["Carrick2015", "CF4"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=True, sample_beta=None,
        zcmb_max=0.05)
    X[simname] = - get_gof("neg_lnZ_harmonic", fname)

# Take 'IndranilVoid_gass' as a reference
ref = X["IndranilVoid_exp"]
for key in X:
    X[key] -= ref

for key in list(X.keys()):
    val = X.pop(key)
    X[simname_to_pretty(key)] = round(val / np.log(10), 2)

X

### Extended analysis, varying the void size

#### Corner plot

In [ ]:
catalogue = ['LOSS', 'Foundation', 'CF4_TFR_i', 'CF4_TFR_notSDSS_w1']
# catalogue = ['LOSS', 'Foundation']

ranges={'void_size':[0.1, 3],
        'rLG_deterministic': [0, None]}

X = []
for simname in ["IndranilVoidSizeVar_exp", "IndranilVoidSizeVar_gauss"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=False, sample_beta=None,
        zcmb_max=0.05)

    X_i = samples_to_getdist(get_samples(fname, True), simname_to_pretty(simname), ranges=ranges)
    # X_i.updateSettings(settings={'mult_bias_correction_order':2 },)
    X.append(X_i)


params = ["Vmag", "l", "b", "rLG_deterministic", "void_size"]
with plt.style.context("science"):
    g = plots.get_subplot_plotter()
    g.settings.figure_legend_frame = False
    g.settings.alpha_filled_add = 0.75
    g.settings.legend_fontsize = 14  # Change this to your desired font size

    g.triangle_plot(X, params=params, filled=True, legend_loc='upper right')
    # plt.gcf().suptitle(simname_to_pretty(simname), y=1.025)
    plt.gcf().tight_layout()
    plt.gcf().show()
    plt.gcf().savefig(f"../../plots/void_extended.pdf", dpi=500, bbox_inches='tight')

#### Evidence comparison

In [ ]:
catalogue = ['LOSS', 'Foundation', 'CF4_TFR_i', 'CF4_TFR_notSDSS_w1']
gof = "BIC"

X = {}
for simname in ["IndranilVoid_exp", "IndranilVoid_gauss", "IndranilVoid_mb", "no_field"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=False, sample_beta=None, no_Vext=None,
        zcmb_max=0.05)

    X[simname] = get_gof(gof, fname)

for simname in ["IndranilVoidSizeVar_exp", "IndranilVoidSizeVar_gauss", "IndranilVoidSizeVar_mb", "no_field"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=False, sample_beta=None, zcmb_max=0.05)

    X[simname] = get_gof(gof, fname)


for simname in ["Carrick2015", "CF4"]:
    fname = paths.flow_validation(
        fdir, simname, catalogue, inference_method="mike",
        sample_alpha=True, sample_beta=None,
        zcmb_max=0.05)
    X[simname] = get_gof(gof, fname)

for key in X:
    X[key] *= -1


# Take 'IndranilVoid_gass' as a reference
ref = X["IndranilVoid_exp"]
for key in X:
    X[key] -= ref

for key in list(X.keys()):
    val = X.pop(key)
    X[simname_to_pretty(key)] = round(val / np.log(10), 2)

X